In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import logging
import os
import warnings
import itertools
from datetime import datetime, timedelta
from functools import partial
from pathlib import Path
from optimization_engines import (
    equal_weights_baseline, 
    genetic_algorithm_qubo, 
    scipy_slsqp_qubo, 
    riskfolio_qubo,
    dwave_cqm_qubo,
)

# Load D-Wave API token from .env file
from dotenv import load_dotenv
load_dotenv()

SEED = 12
np.random.seed(SEED)
msg_level = logging.INFO
# Suppress all RuntimeWarnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

## QUBO Portfolio Optimization using D-Wave Quantum Solver

In [ ]:
# Create a logger
logger = logging.getLogger("inspect_results_logger")
logger.setLevel(msg_level)  # Set the level for this logger

# Create a handler (where to send the logs)
handler = logging.StreamHandler()  # Send to the console
handler.setLevel(msg_level)

# Create a formatter (how to format the logs)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(handler)

### Path definition

In [ ]:
benchmark_path = '../../data/benchmark_gspc.pkl'
source_path = '../../data/stocks_adjclose.pkl'

### Data loading

In [ ]:
benchmark = pd.read_pickle(benchmark_path)
sns.lineplot(benchmark['^GSPC'])
benchmark.head()

In [ ]:
source = pd.read_pickle(source_path)
print(source.shape)
# Check if any row contains at least one NaN
#print(source_wide.isnull().any(axis=1))
# Check if any value in the DataFrame is null
has_any_nan = source.isnull().values.any()
print("Any NaN in source_wide:", has_any_nan)
source.head()

### Correlation Analysis

* Determine a set of stocks with minimal correlation

In [ ]:
df_corr = source.corr()
df_corr.head()

#### Rank correlation ascending

In [ ]:
# rank by correlation
corr_sum = df_corr.map(lambda x: abs(x)).sum()
corr_rank = corr_sum.sort_values().rank(method='min').astype(int)
corr_rank

#### Rank returns descending

In [ ]:
# rank by returns
return_rank = source.diff().sum(axis=0).sort_values().rank(method='min', ascending=False).astype(int)
return_rank

### Portfolio Stats

#### Select sets of 10 and 100 stocks with maximal returns and minimal correlation

In [ ]:
select_10 = (return_rank + corr_rank).sort_values().reset_index()['Ticker'].values[:11]
select_10

In [ ]:
select_100 = (return_rank + corr_rank).sort_values().reset_index()['Ticker'].values[:101]
select_100

Defining Fitness Function for the calculation of Sharpe Ratio. I am using the Portfolio Variance as 

Portfolio variance = w12σ12 + w22σ22 + 2w1w2Cov1,2

In [ ]:
def portfolio_stats(weights, data):
    logger.info("=== portfolio_stats START ===")
    logger.info(f"Input weights shape: {np.array(weights).shape}")
    logger.info(f"Input data shape: {data.shape}")
    
    weights = np.array(weights)
    logger.info("Calculating log returns...")
    returns = np.log(data) - np.log(data.shift(1)) # log return to minimize fp error
    logger.info(f"Log returns calculated, shape: {returns.shape}")
    
    # CONSISTENT: Always use log returns, never pct_change()
    logger.info("Calculating portfolio return...")
    port_return = np.sum(returns.mean() * weights) 
    logger.info(f"Portfolio return: {port_return}")
    
    logger.info("Calculating portfolio volatility...")
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() , weights)))
    logger.info(f"Portfolio volatility: {port_vol}")
    
    try:
        sharpe_ratio = port_return/port_vol
        logger.info(f"Sharpe ratio: {sharpe_ratio}")
    except Exception as e:
        logger.warning(f"Sharpe ratio calculation failed: {e}")
        sharpe_ratio = 0
    
    logger.info("=== portfolio_stats END ===")
    return sharpe_ratio, port_return, port_vol

### Data generator

Data generator to instantiate blocks on demand

In [ ]:
def generate_data(df, benchmark, days_to_avg=30, days_to_opt=30, seed=SEED):
    df2 = df.reset_index()
    benchmark2 = benchmark.reset_index()
    elements = df2.sample(n=100, random_state=seed).index # definint a maximum of 100 different sampled initial dates
    for idx in elements:
        df_sample = df2.iloc[idx-days_to_avg:idx+days_to_opt, :]
        df_sample = df_sample.set_index('ds')
        df_sample_b = benchmark2.iloc[idx-days_to_avg:idx+days_to_opt, :]
        df_sample_b = df_sample_b.set_index('ds').drop(['index'], axis=1)
        yield df_sample, df_sample_b

### Backtest

In [ ]:
def backtest(optimization_function, data, benchmark, initial_capital, avg_period, opt_period):
    logger.info("="*80)
    logger.info("=== BACKTEST START ===")
    logger.info(f"Data shape: {data.shape}, Benchmark shape: {benchmark.shape}")
    logger.info(f"Initial capital: {initial_capital}, Avg period: {avg_period}, Opt period: {opt_period}")
    
    portfolio_value = initial_capital
    portfolio_returns = []
    benchmark_returns = []
    portfolio_total_return = []
    portfolio_sharpe_ratios = []
    weights_history = pd.DataFrame(index=data.index, columns=data.columns)
    portfolio_value_history = pd.Series(index=data.index, name='Portfolio Value', dtype='float')
    portfolio_value_history.iloc[0] = portfolio_value

    logger.info(f"Starting backtest loop for {opt_period} iterations...")
    j = 0
    for i in range(avg_period+1, avg_period + opt_period+1):
        logger.info(f"\n--- Iteration {i-avg_period}/{opt_period} (j={j}, i={i}) ---")
        
        df = data.iloc[j:i, :]
        logger.info(f"Data slice shape: {df.shape}, Date range: {df.index[0]} to {df.index[-1]}")
        
        # Use log returns for consistency with optimization functions
        logger.info("Calculating log returns for slice...")
        df_log_returns = np.log(df) - np.log(df.shift(1))
        df_log_returns = df_log_returns.dropna(axis=0)
        logger.info(f"Log returns shape after dropna: {df_log_returns.shape}")
        
        #logger.debug(f'df_log_returns: {df_log_returns}')
        weights = optimization_function(df)
        logger.info(f"Optimization completed. Weights shape: {weights.shape}, Sum: {weights.sum():.6f}")
        logger.info(f"Weights min: {weights.min():.6f}, max: {weights.max():.6f}, non-zero count: {(weights > 0).sum()}")
        
        weights[weights < 0] = 0
        weights /= weights.sum()
        logger.info(f"Weights after normalization - Sum: {weights.sum():.6f}")
        
        weights_history.loc[df.index[-1]] = weights
        #sharpe_ratio, portfolio_return, portfolio_volatility = portfolio_stats(weights, df)
        # portfolio_change = df.iloc[-2:, :].pct_change() * weights
        logger.info("Calculating portfolio change...")
        portfolio_change = df_log_returns.iloc[-1] * weights
        #portfolio_return = portfolio_change.sum(axis=1).iloc[-1]
        portfolio_return = portfolio_change.sum()
        logger.info(f"Portfolio return for this period: {portfolio_return:.6f}")
        
        portfolio_returns.append(portfolio_return)
        # print(f'portfolio returns: {portfolio_returns}')
        # Use log returns for benchmark as well
        logger.info("Calculating benchmark returns...")
        benchmark_log_returns = np.log(benchmark.iloc[j:i, :]) - np.log(benchmark.iloc[j:i, :].shift(1))
        benchmark_return = benchmark_log_returns.iloc[-1].values.tolist()[0]
        logger.info(f"Benchmark return for this period: {benchmark_return:.6f}")
        
        benchmark_returns.append(benchmark_return)
        # print(f'benchmark_returns: {benchmark_returns}')
        logger.info("Calculating cumulative returns...")
        portfolio_cumulative_returns = np.cumprod([k + 1 for k in portfolio_returns])
        # print(f'portfolio cumulative returns: {portfolio_cumulative_returns}')
        benchmark_cumulative_returns = np.cumprod([k + 1 for k in  benchmark_returns])
        # print(f'benchmark cumulative returns: {benchmark_cumulative_returns}')
        portfolio_mean_return = np.mean(portfolio_returns)
        benchmark_mean_return = np.mean(benchmark_returns)
        portfolio_volatility = np.std(portfolio_returns) 
        benchmark_volatility = np.std(benchmark_returns)
        logger.info(f"Portfolio mean return: {portfolio_mean_return:.6f}, volatility: {portfolio_volatility:.6f}")
        logger.info(f"Benchmark mean return: {benchmark_mean_return:.6f}, volatility: {benchmark_volatility:.6f}")
        
        try:
            sharpe_ratio = (portfolio_mean_return) / portfolio_volatility
            logger.info(f"Sharpe ratio: {sharpe_ratio:.6f}")
        except Exception as e:
            logger.warning(f"Sharpe ratio calculation failed: {e}")
            sharpe_ratio = 0
        portfolio_sharpe_ratios.append(sharpe_ratio)

         # Portfolio & Benchmark value
        benchmark_value = initial_capital * benchmark_cumulative_returns[-1]
        portfolio_value = initial_capital * portfolio_cumulative_returns[-1]
        logger.info(f"Portfolio value: ${portfolio_value:,.2f}, Benchmark value: ${benchmark_value:,.2f}")
        j += 1

    logger.info("\nCalculating final cumulative returns...")
    portfolio_cumulative_returns = portfolio_cumulative_returns - portfolio_cumulative_returns[0]
    benchmark_cumulative_returns = benchmark_cumulative_returns - benchmark_cumulative_returns[0]
    logger.info(f"Final portfolio cumulative return: {portfolio_cumulative_returns[-1]:.6f}")
    logger.info(f"Final benchmark cumulative return: {benchmark_cumulative_returns[-1]:.6f}")

    # Plot the results
    logger.info("Creating plot...")
    plt.figure(figsize=(12, 6))
    plt.plot(portfolio_cumulative_returns, label='Portfolio')
    plt.plot(benchmark_cumulative_returns, label='Benchmark')
    #plt.plot(portfolio_returns, label='Portfolio')
    #plt.plot(benchmark_returns, label='Benchmark')
    plt.legend(loc='upper left')
    plt.title('Backtesting Results')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Returns')
    plt.show()

    logger.info("=== BACKTEST END ===")
    logger.info("="*80)
    return weights_history, portfolio_value_history, portfolio_cumulative_returns, benchmark_cumulative_returns

In [ ]:
import pickle

def write_pickle_dict(data, file_path):
    """Pickles a dictionary and saves it to a file."""
    try:
        with open(file_path, 'wb') as f:  # Open the file in binary write mode ('wb')
            pickle.dump(data, f)
        print(f"Dictionary pickled and saved to {file_path}")
    except Exception as e:
        print(f"An error occurred while pickling: {e}")

def read_pickle_dict(file_path):
    try:
        with open(file_path, 'rb') as f:
            loaded_dict = pickle.load(f)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")
    return loaded_dict

### Run Experiment Function

In [ ]:
def run_experiment(results_path_template, data, benchmark, opt_fun, parameters):
    logger.info("\n" + "#"*100)
    logger.info("### RUN EXPERIMENT START ###")
    logger.info("#"*100)
    logger.info(f"Results path template: {results_path_template}")
    logger.info(f"Data shape: {data.shape}")
    logger.info(f"Benchmark shape: {benchmark.shape}")
    logger.info(f"Parameters: {parameters}")

    n_periods = parameters['n_periods']
    days_to_avg = parameters['days_to_avg']
    days_to_opt = parameters['days_to_opt']
    initial_capital = parameters['initial_capital']
    
    logger.info(f"Creating data generator with n_periods={n_periods}...")
    datagen = generate_data(data, benchmark)

        
    for i in range(n_periods):
        logger.info(f"\n{'='*80}")
        logger.info(f"PERIOD {i+1}/{n_periods}")
        logger.info(f"{'='*80}")
        
        results_path = results_path_template.format(i)
        logger.info(f"Results path: {results_path}")
        
        if not os.path.exists(results_path):
            logger.info(f"Results file does not exist, running optimization...")
            try:
                logger.info("Getting next data sample from generator...")
                df, df_b = next(datagen)
                logger.info(f"Data sample retrieved - df shape: {df.shape}, benchmark shape: {df_b.shape}")
                print(f'initial date: {df.iloc[days_to_avg+1:, :].index[0]}')
                
                logger.info("Creating source data plot...")
                plt.figure(figsize=(12, 6))
                plt.plot(df.iloc[days_to_avg+1:days_to_avg+days_to_opt+1, :].sum(axis=1), label='Source')
                plt.plot(df_b, label='Benchmark')
                plt.legend(loc='upper left')
                plt.title('Source Data')
                plt.xlabel('Date')
                plt.ylabel('Stock Values')
                plt.show()
                
                logger.info(f"Starting backtest at {datetime.now()}")
                start_time = datetime.now()
                
                weights_history, portfolio_value_history, portfolio_cumulative_returns, benchmark_cumulative_returns = backtest(opt_fun, df, df_b, initial_capital=initial_capital, avg_period=days_to_avg,opt_period=days_to_opt)
                
                end_time = datetime.now()
                dt = abs(end_time - start_time)
                logger.info(f'Backtest completed at {end_time}')
                logger.info(f'Total backtest time: {dt.total_seconds():.2f} seconds')
                logger.debug(f'portfolio cumulative returns: {portfolio_cumulative_returns}')
                
            except Exception as e: 
                logger.error(f'FAILED {results_path} due to {e}', exc_info=True)
                print(f'Failed {results_path} due to {e}')
                weights_history, portfolio_value_history, portfolio_cumulative_returns, benchmark_cumulative_returns = [], [], [], []
                dt = timedelta(0)

            logger.info("Saving results to pickle file...")
            result = {
                "round": i, 
                "start_date": df.index[0],
                "end_date": df.index[-1],
                "days_to_avg": days_to_avg,
                "days_to_opt": days_to_opt,
                "weights_history": weights_history,
                "portfolio_value_history": portfolio_value_history,
                "portfolio_cumulative_returns": portfolio_cumulative_returns,
                "benchmark_cumulative_returns": benchmark_cumulative_returns,
                "total_run_time": dt.total_seconds()
                }
            write_pickle_dict(result, results_path)
            logger.info(f"Results saved successfully to {results_path}")

        else:
            logger.info(f"Results file already exists, loading from {results_path}...")
            results = read_pickle_dict(results_path)
            portfolio_cumulative_returns = results['portfolio_cumulative_returns']
            benchmark_cumulative_returns = results['benchmark_cumulative_returns']
            logger.info(f"Results loaded - portfolio return: {portfolio_cumulative_returns[-1] if len(portfolio_cumulative_returns) > 0 else 'N/A'}")

            # Plot the results
            logger.info("Creating results plot...")
            plt.figure(figsize=(12, 6))
            plt.plot(portfolio_cumulative_returns, label='Portfolio')
            plt.plot(benchmark_cumulative_returns, label='Benchmark')
            plt.legend(loc='upper left')
            plt.title('Backtesting Results')
            plt.xlabel('Date')
            plt.ylabel('Cumulative Returns')
            plt.show()
    
    logger.info("\n" + "#"*100)
    logger.info("### RUN EXPERIMENT END ###")
    logger.info("#"*100 + "\n")
    return None

### Performance Analysis Functions

In [ ]:
def performance_summary(solver_configs, n_periods):
    """Generate performance summary across all periods for multiple solvers."""
    summary_data = []
    
    for config in solver_configs:
        solver_name = config['name']
        path_template = config['path_template']
        
        returns_data = []
        execution_times = []
        successful_periods = 0
        
        for period in range(n_periods):
            file_path = path_template.format(period)
            try:
                result_data = read_pickle_dict(file_path)
                if result_data is not None:
                    portfolio_returns = result_data.get('portfolio_cumulative_returns', [])
                    execution_time = result_data.get('total_run_time', 0)
                    
                    # Safe check for portfolio returns data
                    try:
                        if portfolio_returns is not None and len(portfolio_returns) > 0:
                            final_return = portfolio_returns[-1]
                            returns_data.append(final_return)
                            execution_times.append(execution_time)
                            successful_periods += 1
                    except (TypeError, IndexError):
                        pass
            except Exception as e:
                logger.debug(f'Could not process {file_path}: {e}')
        
        if returns_data:
            summary_data.append({
                'Solver': solver_name,
                'Periods': successful_periods,
                'Avg_Return': np.mean(returns_data),
                'Std_Return': np.std(returns_data),
                'Min_Return': np.min(returns_data),
                'Max_Return': np.max(returns_data),
                'Avg_Time': np.mean(execution_times),
                'Std_Time': np.std(execution_times),
                'Success_Rate': successful_periods / n_periods * 100
            })
    
    if summary_data:
        df = pd.DataFrame(summary_data)
        df = df.round(4)
        return df
    else:
        return pd.DataFrame()

### Global Parameters

QUBO-specific parameters for quantum optimization

In [ ]:
parameters = {
    "n_periods": 1,
    "days_to_avg": 30,
    "days_to_opt": 30,
    "budget": 1_000_000.0,
}

In [ ]:
# D-Wave CQM QUBO Optimization - Define the optimization function
opt_fun_dwave_cqm = partial(dwave_cqm_qubo, budget=parameters["budget"])

# Extract n_periods for use in performance analysis
n_periods = parameters['n_periods']

print(f"D-Wave CQM QUBO optimization function created with parameters:")
# print(f"  budget: {parameters['budget']}")
print(f"  n_periods: {n_periods}")

### Test 1: D-Wave CQM QUBO Optimization with 100 stocks 

In [ ]:
# df_corr = source.corr()
# corr_sum = df_corr.map(lambda x: abs(x)).sum()
# corr_rank = corr_sum.sort_values().rank(method='min').astype(int)
# return_rank = source.diff().sum(axis=0).sort_values().rank(method='min', ascending=False).astype(int)
# select_100 = (return_rank + corr_rank).sort_values().reset_index()['Ticker'].values[:101]
# data_100 = source[select_100]

In [ ]:
# results_path_dwave_cqm_100 = '../../results/qubo_unrestricted/results_100_{}_dwave_cqm.pkl'
# _ = run_experiment(results_path_dwave_cqm_100, data_100, benchmark, opt_fun_dwave_cqm, parameters)

### Test 2: D-Wave CQM QUBO Optimization with Full Dataset

In [ ]:
results_path_dwave_cqm_full = '../../results/qubo_unrestricted/results_full_{}_dwave_cqm.pkl'
_ = run_experiment(results_path_dwave_cqm_full, source, benchmark, opt_fun_dwave_cqm, parameters)

### Test 3: Wide Dataset (Stocks + ETFs)

In [ ]:
%%script false --no-raise-error
source_path_extended = '../../data/etfs_close.pkl'

In [ ]:
%%script false --no-raise-error
source_extended = pd.read_pickle(source_path_extended)
source_extended.head()

In [ ]:
%%script false --no-raise-error
source_wide = pd.merge(source, source_extended, on='ds', how='left').set_index('ds').dropna()
print(source_wide.shape)
# Check if any row contains at least one NaN
#print(source_wide.isnull().any(axis=1))
# Check if any value in the DataFrame is null
has_any_nan = source_wide.isnull().values.any()
print("Any NaN in source_wide:", has_any_nan)
source_wide.head()

#### D-Wave CQM QUBO Optimization - Wide Dataset

In [ ]:
%%script false --no-raise-error
results_path_dwave_cqm_wide = '../../results/qubo_unrestricted/results_wide_{}_dwave_cqm.pkl'
_ = run_experiment(results_path_dwave_cqm_wide, source_wide, benchmark, opt_fun_dwave_cqm, parameters)

#### Performance Analysis for Wide Dataset

In [ ]:
%%script false --no-raise-error
# Performance analysis for D-Wave CQM QUBO optimization (Wide Dataset - Stocks + ETFs)
dwave_cqm_wide_solvers = [
    {'path_template': '../../results/qubo_unrestricted/results_wide_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM QUBO'},
]

print("\n" + "="*100)
print("COMPREHENSIVE PERFORMANCE SUMMARY - D-Wave CQM QUBO Optimization (Wide Dataset - Stocks + ETFs)")
print("="*100)
summary_df = performance_summary(dwave_cqm_wide_solvers, n_periods)
if not summary_df.empty:
    display(summary_df)
    
    # Performance analysis
    print("\nPerformance Analysis:")
    print("-" * 50)
    print(f"Successful periods: {summary_df.iloc[0]['Periods']}/{n_periods}")
    print(f"Average return: {summary_df.iloc[0]['Avg_Return']:.4f}")
    print(f"Average execution time: {summary_df.iloc[0]['Avg_Time']:.2f}s")
    print(f"Success rate: {summary_df.iloc[0]['Success_Rate']:.1f}%")
else:
    print("No summary data available")

### Test 5: Synthetic Dataset (1,200 columns)

Test with the expanded synthetic dataset to evaluate scalability

In [ ]:
%%script false --no-raise-error
source_path_synthetic = '../../data/synthetic_close.pkl'

In [ ]:
%%script false --no-raise-error
source_synthetic = pd.read_pickle(source_path_synthetic)
print(source_synthetic.shape)
# Check if any value in the DataFrame is null
has_any_nan = source_synthetic.isnull().values.any()
print("Any NaN in source_synthetic:", has_any_nan)
source_synthetic.head()

In [ ]:
%%script false --no-raise-error
df_synth = source_synthetic
df_synth.set_index('ds', inplace=True)
days_to_avg = parameters['days_to_avg']
display(df_synth.head())
has_any_nan = source_synthetic.isnull().values.any()
print("Any NaN in source_synth:", has_any_nan)
print(f'initial date: {df_synth.iloc[days_to_avg+1, :].index}')

#### D-Wave CQM QUBO Optimization - Synthetic Dataset

In [ ]:
%%script false --no-raise-error
results_path_dwave_cqm_synthetic = '../../results/qubo_unrestricted/results_synthetic1200_{}_dwave_cqm.pkl'
_ = run_experiment(results_path_dwave_cqm_synthetic, df_synth, benchmark, opt_fun_dwave_cqm, parameters)

#### Performance Analysis for Synthetic Dataset

In [ ]:
%%script false --no-raise-error
# Performance analysis for D-Wave CQM QUBO optimization (Synthetic Dataset - 10,000 columns)
dwave_cqm_synthetic_solvers = [
    {'path_template': '../../results/qubo_unrestricted/results_synthetic_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM QUBO (Synthetic)'},
]

print("\n" + "="*100)
print("COMPREHENSIVE PERFORMANCE SUMMARY - D-Wave CQM QUBO Optimization (Synthetic Dataset - 10,000 columns)")
print("="*100)
summary_df = performance_summary(dwave_cqm_synthetic_solvers, n_periods)
if not summary_df.empty:
    display(summary_df)
    
    # Performance analysis
    print("\nPerformance Analysis:")
    print("-" * 50)
    print(f"Successful periods: {summary_df.iloc[0]['Periods']}/{n_periods}")
    print(f"Average return: {summary_df.iloc[0]['Avg_Return']:.4f}")
    print(f"Average execution time: {summary_df.iloc[0]['Avg_Time']:.2f}s")
    print(f"Success rate: {summary_df.iloc[0]['Success_Rate']:.1f}%")
else:
    print("No summary data available")

## Overall QUBO Performance Summary

In [ ]:
# Compare all QUBO optimization methods
all_qubo_solvers = [
    {'path_template': '../../results/qubo_optimization/results_full_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM QUBO'},
    {'path_template': '../../results/qubo_optimization/results_full_{}_genetic.pkl', 'name': 'Genetic Algorithm QUBO'},
    {'path_template': '../../results/qubo_optimization/results_full_{}_scipy.pkl', 'name': 'SciPy SLSQP QUBO'},
    {'path_template': '../../results/qubo_optimization/results_full_{}_equal_weights.pkl', 'name': 'Equal Weights Baseline'},
    {'path_template': '../../results/qubo_unrestricted/results_wide_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM QUBO (Wide)'},
    #{'path_template': '../../results/qubo_unrestricted/results_synthetic_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM QUBO (Synthetic)'},
]

print("\n" + "="*120)
print("OVERALL QUBO OPTIMIZATION PERFORMANCE COMPARISON - ALL METHODS")
print("="*120)

summary_df = performance_summary(all_qubo_solvers, n_periods)
if not summary_df.empty:
    # Sort by average return for better readability
    summary_df_sorted = summary_df.sort_values('Avg_Return', ascending=False)
    display(summary_df_sorted)
    
    # Performance analysis
    print("\nPerformance Analysis Across All QUBO Methods:")
    print("-" * 70)
    if not summary_df_sorted.empty:
        print(f"Best performing method: {summary_df_sorted.iloc[0]['Solver']} (avg return: {summary_df_sorted.iloc[0]['Avg_Return']:.4f})")
        print(f"Fastest execution: {summary_df.loc[summary_df['Avg_Time'].idxmin(), 'Solver']} ({summary_df['Avg_Time'].min():.2f}s)")
        print(f"Most reliable: {summary_df.loc[summary_df['Success_Rate'].idxmax(), 'Solver']} ({summary_df['Success_Rate'].max():.1f}% success rate)")
        
        # Method comparison
        print("\nMethod Comparison:")
        print("-" * 90)
        for _, row in summary_df_sorted.iterrows():
            if "Wide" in row['Solver']:
                dataset_info = "680 assets"
            elif "Synthetic" in row['Solver']:
                dataset_info = "10,000 assets"
            else:
                dataset_info = "439 assets"
            print(f"{row['Solver']:<30}: {dataset_info:>12} | {row['Avg_Time']:>8.2f}s | {row['Success_Rate']:>6.1f}% | {row['Avg_Return']:>8.4f}")
    else:
        print("No performance data available")
        
else:
    print("No summary data available")